In [ ]:
project_tables = [
    "workspace.healthcare.bronze_patients",
    "workspace.healthcare.bronze_doctors",
    "workspace.healthcare.bronze_admissions",
    "workspace.healthcare.silver_patients",
    "workspace.healthcare.silver_doctors",
    "workspace.healthcare.silver_admissions",
    "workspace.healthcare.silver_hospital_records",
    "workspace.healthcare.gold_department_performance",
    "workspace.healthcare.gold_disease_summary",
    "workspace.healthcare.gold_patient_summary",
    "workspace.healthcare.gold_doctor_performance",
    "workspace.healthcare.gold_hospital_kpis",
    "workspace.healthcare.gold_data_quality_report",
    "workspace.healthcare.pipeline_audit_log"
]

In [ ]:
for table_name in project_tables:
    try:
        record_count = spark.table(
            table_name
        ).count()
        print(
            f"SUCCESS | {table_name} | "
            f"Records: {record_count}"
        )
    except Exception as e:
        print(
            f"FAILED | {table_name} | "
            f"Error: {str(e)}"
        )

SUCCESS | workspace.healthcare.bronze_patients | Records: 10
SUCCESS | workspace.healthcare.bronze_doctors | Records: 5
SUCCESS | workspace.healthcare.bronze_admissions | Records: 5
SUCCESS | workspace.healthcare.silver_patients | Records: 10
SUCCESS | workspace.healthcare.silver_doctors | Records: 5
SUCCESS | workspace.healthcare.silver_admissions | Records: 5
SUCCESS | workspace.healthcare.silver_hospital_records | Records: 5
SUCCESS | workspace.healthcare.gold_department_performance | Records: 5
SUCCESS | workspace.healthcare.gold_disease_summary | Records: 5
SUCCESS | workspace.healthcare.gold_patient_summary | Records: 5
SUCCESS | workspace.healthcare.gold_doctor_performance | Records: 5
SUCCESS | workspace.healthcare.gold_hospital_kpis | Records: 1
SUCCESS | workspace.healthcare.gold_data_quality_report | Records: 5
SUCCESS | workspace.healthcare.pipeline_audit_log | Records: 1


In [ ]:
from pyspark.sql import Row

In [ ]:
table_status_data = []

for table_name in project_tables:
    try:
        record_count = spark.table(
            table_name
        ).count()
        table_status_data.append(
            Row(
                table_name=table_name,
                record_count=record_count,
                status="SUCCESS"
            )
        )
    except Exception as e:
        table_status_data.append(
            Row(
                table_name=table_name,
                record_count=0,
                status="FAILED"
            )
        )

In [ ]:
project_status = spark.createDataFrame(
    table_status_data
)

display(project_status)

table_name,record_count,status
workspace.healthcare.bronze_patients,10,SUCCESS
workspace.healthcare.bronze_doctors,5,SUCCESS
workspace.healthcare.bronze_admissions,5,SUCCESS
workspace.healthcare.silver_patients,10,SUCCESS
workspace.healthcare.silver_doctors,5,SUCCESS
workspace.healthcare.silver_admissions,5,SUCCESS
workspace.healthcare.silver_hospital_records,5,SUCCESS
workspace.healthcare.gold_department_performance,5,SUCCESS
workspace.healthcare.gold_disease_summary,5,SUCCESS
workspace.healthcare.gold_patient_summary,5,SUCCESS


In [ ]:
from pyspark.sql import Row

In [ ]:
table_status_data = []

for table_name in project_tables:
    try:
        record_count = spark.table(
            table_name
        ).count()
        table_status_data.append(
            Row(
                table_name=table_name,
                record_count=record_count,
                status="SUCCESS"
            )
        )
    except Exception as e:
        table_status_data.append(
            Row(
                table_name=table_name,
                record_count=0,
                status="FAILED"
            )
        )

In [ ]:
project_status = spark.createDataFrame(
    table_status_data
)

display(project_status)

table_name,record_count,status
workspace.healthcare.bronze_patients,10,SUCCESS
workspace.healthcare.bronze_doctors,5,SUCCESS
workspace.healthcare.bronze_admissions,5,SUCCESS
workspace.healthcare.silver_patients,10,SUCCESS
workspace.healthcare.silver_doctors,5,SUCCESS
workspace.healthcare.silver_admissions,5,SUCCESS
workspace.healthcare.silver_hospital_records,5,SUCCESS
workspace.healthcare.gold_department_performance,5,SUCCESS
workspace.healthcare.gold_disease_summary,5,SUCCESS
workspace.healthcare.gold_patient_summary,5,SUCCESS


In [ ]:
failed_tables = project_status.filter(
    project_status.status == "FAILED"
)

failed_count = failed_tables.count()

if failed_count == 0:
    print(
        "🎉 PROJECT PIPELINE VERIFIED SUCCESSFULLY!"
    )
else:
    print(
        f"⚠️ WARNING: {failed_count} table(s) failed verification."
    )
    display(failed_tables)

🎉 PROJECT PIPELINE VERIFIED SUCCESSFULLY!


In [ ]:
hospital_records = spark.table(
    "workspace.healthcare.silver_hospital_records"
)

In [ ]:
from pyspark.sql.functions import (
    col,
    countDistinct,
    avg,
    round
)

In [ ]:
duplicate_admissions = hospital_records.groupBy(
    "admission_id"
).count().filter(
    col("count") > 1
)

display(duplicate_admissions)

admission_id,count


In [ ]:
invalid_age = hospital_records.filter(
    (col("age") < 0) |
    (col("age") > 120)
)

display(invalid_age)

admission_id,patient_id,first_name,last_name,gender,age,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay


In [ ]:
invalid_stays = hospital_records.filter(
    col("length_of_stay") < 0
)

display(invalid_stays)

admission_id,patient_id,first_name,last_name,gender,age,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay


In [ ]:
missing_doctors = hospital_records.filter(
    col("doctor_name").isNull()
)

display(missing_doctors)

admission_id,patient_id,first_name,last_name,gender,age,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay


In [ ]:
hospital_kpis = spark.table(
    "workspace.healthcare.gold_hospital_kpis"
)

display(hospital_kpis)

total_patients,total_admissions,average_patient_age,average_length_of_stay
5,5,40.0,4.6


In [ ]:
department_performance = spark.table(
    "workspace.healthcare.gold_department_performance"
)

display(department_performance)

department,total_patients,total_admissions,average_patient_age,average_length_of_stay
Orthopedics,1,1,48.0,5.0
General Medicine,1,1,44.0,3.0
Pediatrics,1,1,31.0,4.0
Cardiology,1,1,41.0,5.0
Neurology,1,1,36.0,6.0


In [ ]:
disease_summary = spark.table(
    "workspace.healthcare.gold_disease_summary"
)

display(disease_summary)

diagnosis,total_patients,total_admissions,average_length_of_stay
Migraine,1,1,6.0
Heart Disease,1,1,5.0
Diabetes,1,1,3.0
Viral Fever,1,1,4.0
Bone Fracture,1,1,5.0


In [ ]:
doctor_performance = spark.table(
    "workspace.healthcare.gold_doctor_performance"
)

display(doctor_performance)

doctor_id,doctor_name,department,specialization,total_patients,total_admissions,average_length_of_stay
D003,Dr. Amit Verma,Orthopedics,Orthopedic Surgeon,1,1,5.0
D002,Dr. Priya Sharma,Neurology,Neurologist,1,1,6.0
D004,Dr. Sneha Reddy,Pediatrics,Pediatrician,1,1,4.0
D001,Dr. Rajesh Kumar,Cardiology,Cardiologist,1,1,5.0
D005,Dr. Arjun Rao,General Medicine,General Physician,1,1,3.0


In [ ]:
audit_log = spark.table(
    "workspace.healthcare.pipeline_audit_log"
)

display(audit_log)

pipeline_name,target_table,status,execution_timestamp
Healthcare Incremental Patient Load,workspace.healthcare.silver_patients,SUCCESS,2026-09-07T06:05:31.106Z
